In [1]:
import sys
sys.path.append("/home/jovyan/work")

from src.utils.spark_session import get_spark_session
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType, StringType,
    DateType, TimestampType, BooleanType, DecimalType
)
from delta.tables import DeltaTable
from datetime import date, datetime, timedelta
from decimal import Decimal

spark = get_spark_session("build-gold-layer")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

In [2]:
start_date = date(2025, 1, 1)
end_date = date(2027, 12, 31)

date_schema = StructType([
    StructField("date_key", IntegerType(), False),
    StructField("full_date", DateType(), False),
    StructField("day_of_week", StringType(), True),
    StructField("week_of_year", IntegerType(), True),
    StructField("month_no", IntegerType(), True),
    StructField("month_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("is_weekend", BooleanType(), True),
])

date_rows = []
d = start_date
while d <= end_date:
    date_rows.append(Row(
        date_key=int(d.strftime("%Y%m%d")),
        full_date=d,
        day_of_week=d.strftime("%A"),
        week_of_year=d.isocalendar()[1],
        month_no=d.month,
        month_name=d.strftime("%B"),
        quarter=(d.month - 1) // 3 + 1,
        year=d.year,
        is_weekend=d.weekday() >= 5,
    ))
    d += timedelta(days=1)

df_dim_date = spark.createDataFrame(date_rows, schema=date_schema)
df_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

print(f"gold.dim_date: {spark.table('gold.dim_date').count()} rows")
spark.table("gold.dim_date").orderBy("full_date").limit(3).toPandas()

gold.dim_date: 1095 rows


,date_key,full_date,day_of_week,week_of_year,month_no,month_name,quarter,year,is_weekend
0,20250101,2025-01-01,Wednesday,1,1,January,1,2025,False
1,20250102,2025-01-02,Thursday,1,1,January,1,2025,False
2,20250103,2025-01-03,Friday,1,1,January,1,2025,False


In [3]:
vehicle_rows = spark.table("silver.vehicle").filter("is_deleted = false").select(
    "vehicle_id", "vin", "make", "model", "model_year", "branch_id",
    F.col("current_stage").alias("stage"),
    F.col("current_status").alias("status"),
    "created_at",
).collect()

dim_vehicle_schema = StructType([
    StructField("vehicle_key", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("vin", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("branch_id", LongType(), True),
    StructField("stage", StringType(), True),
    StructField("status", StringType(), True),
    StructField("effective_from", TimestampType(), False),
    StructField("effective_to", TimestampType(), True),
    StructField("is_current", BooleanType(), False),
])

dim_vehicle_rows = [
    Row(
        vehicle_key=i, vehicle_id=row.vehicle_id,
        vin=row.vin, make=row.make, model=row.model, model_year=row.model_year,
        branch_id=row.branch_id, stage=row.stage, status=row.status,
        effective_from=row.created_at, effective_to=None, is_current=True,
    )
    for i, row in enumerate(vehicle_rows, start=1)
]

df_dim_vehicle = spark.createDataFrame(dim_vehicle_rows, schema=dim_vehicle_schema)
df_dim_vehicle.write.format("delta").mode("overwrite").saveAsTable("gold.dim_vehicle")

print(f"gold.dim_vehicle: {spark.table('gold.dim_vehicle').count()} rows")
spark.table("gold.dim_vehicle").groupBy("stage", "status").count().show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `silver`.`vehicle` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedRelation [silver, vehicle], [], false


In [ ]:
spark.sql("SHOW DATABASES").show()